# Chapter 7: GRPO — Group Relative Policy Optimization

> **Book:** Agentic AI Engineering  
> **Part:** 2 — RL Methods  
> **System:** Guardian Angel System (GAS) — Multi-agent diabetic patient supervision

In [ ]:
# Install dependencies
# !pip install numpy matplotlib

In [ ]:
import os
TEST_MODE = os.getenv("NOTEBOOK_TEST_MODE", "0") == "1"
print(f"TEST_MODE: {TEST_MODE}")

---
## 🎯 1. Chapter Goal

By the end of this notebook you will:
- Understand **GRPO's group-relative advantage estimation** and why it eliminates the value network
- Implement **GRPO reward computation** with G=8 mock responses
- Apply a **rule-based medical reward function** to score clinical reasoning quality
- Visualize **advantage distributions** across the response group
- Compare **GRPO vs PPO** memory and computational requirements

---
## 📖 2. Theory Recap

### The Problem with PPO's Value Network

PPO requires a **value network** $V_\phi(s)$ to estimate baselines for advantage computation:

$$A_t^{\text{GAE}} = \sum_{l=0}^{\infty} (\gamma\lambda)^l \delta_{t+l}, \quad \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

For LLMs, this value network is typically another copy of the model — **doubling memory requirements**.

### GRPO's Key Insight

GRPO (Shao et al., 2024, used in DeepSeek-R1) eliminates the value network by using **group statistics** as the baseline:

$$A_i = \frac{r_i - \text{mean}(\{r_1, ..., r_G\})}{\text{std}(\{r_1, ..., r_G\}) + \epsilon}$$

Where:
- $G$ = group size (number of responses sampled per prompt)
- $r_i$ = reward for response $i$
- The mean and std serve as the **baseline** (replacing the value network)

### GRPO Objective

$$\mathcal{L}_{\text{GRPO}} = -\mathbb{E}\left[\frac{1}{G}\sum_{i=1}^{G} \min\left(\frac{\pi_\theta(o_i|q)}{\pi_{\text{old}}(o_i|q)} A_i, \text{clip}\left(\frac{\pi_\theta(o_i|q)}{\pi_{\text{old}}(o_i|q)}, 1-\epsilon, 1+\epsilon\right) A_i\right) - \beta D_{KL}(\pi_\theta || \pi_{\text{ref}})\right]$$

### GRPO vs PPO Comparison

| Feature | PPO | GRPO |
|---------|-----|------|
| Value network | ✅ Required | ❌ Not needed |
| Memory overhead | ~40% more | Baseline |
| Advantage estimation | GAE (temporal) | Group normalization |
| Sample efficiency | Higher | Lower (needs G samples) |
| Training stability | Moderate | High |
| Best for | Sequential tasks | Reasoning tasks |

---
## 🔨 3. Build It

### 3.1 Medical Reasoning Task Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(42)

# Medical reasoning task for GAS Advisor
task_prompt = "Given glucose=320 mg/dL, HbA1c=9.2%, what is the likely diagnosis and recommended action?"

print("Medical Reasoning Task:")
print(f"  {task_prompt}")
print()
print("Generating G=8 mock responses (simulating LLM sampling)...")

### 3.2 G=8 Mock Responses

In [ ]:
# G=8 mock responses — varying quality of clinical reasoning
mock_responses = [
    # Response 0: High quality — mentions all key terms
    "The patient likely has hyperglycemia consistent with poorly controlled Type 2 diabetes. "
    "With HbA1c of 9.2%, this indicates chronic hyperglycemia over the past 3 months. "
    "Recommended action: medication review, lifestyle modification including diet and exercise, "
    "and close monitoring. Consider insulin therapy if oral medications are insufficient.",
    
    # Response 1: Low quality — vague
    "Glucose is high. Patient should drink water and rest.",
    
    # Response 2: High quality — comprehensive
    "HbA1c of 9.2% strongly suggests chronic hyperglycemia consistent with diabetes mellitus. "
    "Glucose of 320 mg/dL requires immediate management. Diagnosis: uncontrolled Type 2 diabetes. "
    "Action: urgent medication adjustment, lifestyle intervention, check for ketones to rule out DKA, "
    "and schedule endocrinology follow-up.",
    
    # Response 3: Medium quality — mentions some terms
    "The elevated glucose and HbA1c indicate hyperglycemia. "
    "Consider insulin adjustment and dietary counseling for glycemic control.",
    
    # Response 4: Low quality — dangerous advice
    "Patient has high blood sugar. Take extra insulin immediately.",
    
    # Response 5: High quality — best response
    "Diagnosis: Type 2 diabetes with poor glycemic control. HbA1c 9.2% indicates average glucose "
    "~220 mg/dL over 3 months. Current glucose 320 mg/dL represents acute hyperglycemia. "
    "Recommended actions: (1) Check ketones to rule out DKA, (2) Optimize antidiabetic therapy "
    "(consider adding/adjusting insulin), (3) Lifestyle intervention — diet and exercise, "
    "(4) Glycemic monitoring every 2-4 hours, (5) Endocrinology referral.",
    
    # Response 6: Low quality — minimal
    "Blood sugar is 320 which is dangerous. See a doctor.",
    
    # Response 7: Medium-high quality
    "The combination of glucose 320 and HbA1c 9.2% strongly indicates uncontrolled hyperglycemia "
    "in diabetes. Treatment: optimize antidiabetic therapy, lifestyle intervention including "
    "diet modification and exercise, monitor ketones, and schedule follow-up.",
]

G = len(mock_responses)
print(f"Group size G = {G}")
print("\nResponse lengths (word count):")
for i, r in enumerate(mock_responses):
    print(f"  Response {i}: {len(r.split()):3d} words")

### 3.3 Rule-Based Medical Reward Function

In [ ]:
def score_medical_response(response: str) -> float:
    """
    Rule-based reward function for medical reasoning quality.
    Scores based on clinical terminology, completeness, and safety.
    
    Returns: score in [0, 1]
    """
    score = 0.0
    response_lower = response.lower()
    
    # Clinical terminology (each term adds to score)
    clinical_terms = {
        'hyperglycemia': 0.15,
        'hba1c': 0.12,
        'diabetes': 0.10,
        'lifestyle': 0.08,
        'medication': 0.08,
        'insulin': 0.08,
        'glycemic': 0.10,
        'ketones': 0.10,
        'diet': 0.05,
        'exercise': 0.05,
        'monitor': 0.05,
    }
    
    for term, weight in clinical_terms.items():
        if term in response_lower:
            score += weight
    
    # Length bonus: more complete responses score higher
    word_count = len(response.split())
    if word_count > 50:
        score += 0.10
    elif word_count > 20:
        score += 0.05
    
    # Safety penalty: dangerous advice
    dangerous_phrases = ['take extra insulin immediately', 'drink water and rest']
    for phrase in dangerous_phrases:
        if phrase in response_lower:
            score -= 0.20
    
    return float(np.clip(score, 0.0, 1.0))

# Score all responses
rewards = np.array([score_medical_response(r) for r in mock_responses])

print("Reward scores for each response:")
print("-" * 50)
for i, (r, score) in enumerate(zip(mock_responses, rewards)):
    preview = r[:60] + "..." if len(r) > 60 else r
    quality = "🟢 High" if score > 0.5 else ("🟡 Med" if score > 0.2 else "🔴 Low")
    print(f"  Response {i}: {score:.3f} {quality}")
    print(f"    '{preview}'")

print(f"\nGroup statistics:")
print(f"  Mean reward:  {np.mean(rewards):.4f}")
print(f"  Std reward:   {np.std(rewards):.4f}")
print(f"  Min reward:   {np.min(rewards):.4f}")
print(f"  Max reward:   {np.max(rewards):.4f}")

### 3.4 GRPO Advantage Computation

$$A_i = \frac{r_i - \bar{r}}{\sigma_r + \epsilon}$$

In [ ]:
def compute_grpo_advantages(rewards: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """
    GRPO advantage estimation using group statistics.
    
    A_i = (r_i - mean(r)) / (std(r) + eps)
    
    This replaces the value network from PPO!
    The group mean serves as the baseline.
    """
    mean_r = np.mean(rewards)
    std_r = np.std(rewards)
    advantages = (rewards - mean_r) / (std_r + eps)
    return advantages

advantages = compute_grpo_advantages(rewards)

print("GRPO Advantage Computation:")
print("=" * 55)
print(f"  Group mean reward: {np.mean(rewards):.4f}")
print(f"  Group std reward:  {np.std(rewards):.4f}")
print()
print(f"{'Response':>10} {'Reward':>8} {'Advantage':>10} {'Interpretation':>25}")
print("-" * 60)
for i, (r, a) in enumerate(zip(rewards, advantages)):
    interp = "↑ Reinforce" if a > 0.5 else ("↓ Suppress" if a < -0.5 else "→ Neutral")
    print(f"  Resp {i:>2}:  {r:>6.3f}   {a:>+8.3f}   {interp}")

print(f"\nAdvantage statistics:")
print(f"  Mean: {np.mean(advantages):.6f} (should be ~0)")
print(f"  Std:  {np.std(advantages):.6f} (should be ~1)")

### 3.5 Visualization: Reward and Advantage Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

response_labels = [f"R{i}" for i in range(G)]
reward_colors = ['#2ecc71' if r > np.mean(rewards) else '#e74c3c' for r in rewards]
adv_colors = ['#2ecc71' if a > 0 else '#e74c3c' for a in advantages]

# Plot 1: Raw rewards
ax1 = axes[0]
bars1 = ax1.bar(response_labels, rewards, color=reward_colors, edgecolor='black', linewidth=0.5)
ax1.axhline(y=np.mean(rewards), color='navy', linewidth=2, linestyle='--', label=f'Mean={np.mean(rewards):.3f}')
ax1.set_title('Raw Rewards per Response', fontsize=12, fontweight='bold')
ax1.set_ylabel('Reward Score')
ax1.set_xlabel('Response')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0, 1.1)

# Plot 2: GRPO advantages
ax2 = axes[1]
bars2 = ax2.bar(response_labels, advantages, color=adv_colors, edgecolor='black', linewidth=0.5)
ax2.axhline(y=0, color='black', linewidth=1.5, linestyle='--')
ax2.set_title('GRPO Advantages A_i = (r_i - mean) / std', fontsize=12, fontweight='bold')
ax2.set_ylabel('Advantage')
ax2.set_xlabel('Response')
ax2.grid(axis='y', alpha=0.3)
green_patch = mpatches.Patch(color='#2ecc71', label='Reinforce (A > 0)')
red_patch = mpatches.Patch(color='#e74c3c', label='Suppress (A < 0)')
ax2.legend(handles=[green_patch, red_patch])

# Plot 3: Reward distribution histogram
ax3 = axes[2]
ax3.hist(rewards, bins=6, color='#3498db', edgecolor='black', alpha=0.7)
ax3.axvline(x=np.mean(rewards), color='red', linewidth=2, linestyle='--', label=f'Mean={np.mean(rewards):.3f}')
ax3.set_title('Reward Distribution (Group of G=8)', fontsize=12, fontweight='bold')
ax3.set_xlabel('Reward Score')
ax3.set_ylabel('Count')
ax3.legend()
ax3.grid(alpha=0.3)

plt.suptitle('GRPO: Group Relative Policy Optimization\nMedical Reasoning Task', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('grpo_advantages.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nKey insight: GRPO uses the GROUP MEAN as the baseline.")
print("Responses above average get positive advantages (reinforced).")
print("Responses below average get negative advantages (suppressed).")
print("No value network needed!")

### 3.6 Memory Savings: GRPO vs PPO

In [ ]:
def estimate_training_memory(model_params_b: float, method: str) -> dict:
    """
    Estimate GPU memory for RL training methods.
    
    Args:
        model_params_b: model size in billions of parameters
        method: 'ppo' or 'grpo'
    """
    bytes_per_param_bf16 = 2
    model_gb = model_params_b * 1e9 * bytes_per_param_bf16 / 1e9
    
    if method == 'ppo':
        components = {
            'Policy model': model_gb,
            'Reference model': model_gb,
            'Value network': model_gb,  # PPO needs this!
            'Optimizer states': model_gb * 2,  # Adam: 2x model
            'Activations': model_gb * 0.5,
        }
    else:  # grpo
        components = {
            'Policy model': model_gb,
            'Reference model': model_gb,
            # No value network!
            'Optimizer states': model_gb * 2,
            'Activations': model_gb * 0.5,
        }
    
    total = sum(components.values())
    return {'components': components, 'total_gb': total}

# Compare for a 7B model
model_size = 7.0  # 7B parameters

ppo_mem = estimate_training_memory(model_size, 'ppo')
grpo_mem = estimate_training_memory(model_size, 'grpo')

print(f"Memory Comparison for {model_size}B parameter model (bf16)")
print("=" * 60)
print(f"\n{'PPO Memory Breakdown':}")
for component, gb in ppo_mem['components'].items():
    print(f"  {component:<20}: {gb:6.1f} GB")
print(f"  {'TOTAL':<20}: {ppo_mem['total_gb']:6.1f} GB")

print(f"\n{'GRPO Memory Breakdown':}")
for component, gb in grpo_mem['components'].items():
    print(f"  {component:<20}: {gb:6.1f} GB")
print(f"  {'TOTAL':<20}: {grpo_mem['total_gb']:6.1f} GB")

savings = ppo_mem['total_gb'] - grpo_mem['total_gb']
savings_pct = savings / ppo_mem['total_gb'] * 100
print(f"\n✅ GRPO saves {savings:.1f} GB ({savings_pct:.0f}% less memory than PPO)")
print(f"   This is the value network that GRPO eliminates!")

---
## 🏥 4. GAS Connection

### GRPO in the Guardian Angel System

The **GAS Advisor Agent** uses GRPO to learn step-by-step clinical reasoning. For each patient state, the agent generates G=8 candidate responses, scores them with a medical reward function, and uses group-normalized advantages to update the policy.

```
GRPO Training Loop for GAS Advisor:

Patient State (glucose, HbA1c, symptoms)
         ↓
  Sample G=8 responses
         ↓
  Score with medical reward:
  - Clinical terminology ✓
  - Safety compliance ✓  
  - Completeness ✓
         ↓
  Compute group advantages: A_i = (r_i - mean) / std
         ↓
  Update policy (reinforce good, suppress bad)
```

**Why GRPO for GAS?**

1. **Memory efficiency**: GAS runs 5 agents concurrently — saving 40% memory per agent is critical
2. **Reasoning quality**: GRPO was used to train DeepSeek-R1's chain-of-thought reasoning
3. **No value network**: Fewer components = fewer failure points in a medical system
4. **Group diversity**: Sampling G=8 responses encourages diverse clinical reasoning paths

**GRPO trains the GAS Advisor to reason step-by-step about patient state.**

The reward function rewards responses that:
- Use correct clinical terminology (hyperglycemia, HbA1c, glycemic control)
- Recommend appropriate interventions (medication review, lifestyle changes)
- Flag emergencies (DKA, severe hypoglycemia)
- Avoid dangerous advice (self-medication, ignoring symptoms)

---
## ✅ 5. Checkpoint

Test your understanding:

**Q1: What does GRPO use instead of a value network?**  
A: GRPO uses the group mean reward as the baseline. The advantage for each response is computed as (r_i - mean(r)) / std(r), where the statistics are computed over the G responses sampled for the same prompt.

**Q2: Why does GRPO require sampling G responses per prompt?**  
A: The group statistics (mean and std) can only be computed if you have multiple responses. With G=1, you'd have no baseline. Typical values are G=4 to G=16.

**Q3: What does a negative advantage mean for a response?**  
A: A negative advantage means the response scored below the group average. The policy update will decrease the probability of generating this response — it's being suppressed.

**Q4: How much memory does GRPO save compared to PPO for a 7B model?**  
A: GRPO saves approximately 14 GB (one full copy of the 7B model in bf16) by eliminating the value network. This is roughly 25-40% of total training memory.

**Q5: What famous model used GRPO for training?**  
A: DeepSeek-R1 used GRPO to train its chain-of-thought reasoning capabilities, demonstrating that group-relative advantages can produce strong reasoning without a value network.